## MiRAGE: 
Mining Relationships for Advanced Generative Evaluation in Drug Repositioning

In [8]:
#主训练/预测流程（含 RandomForest）	模型训练
import pandas as pd
from tqdm import tqdm

In [9]:
# --- Cell 1: DDCD 配置 (符合 MiRAGE 论文逻辑, BIB 2024) ---
# 论文: https://academic.oup.com/bib/article/25/4/bbae337/7717951
#
#  特征维度 (共 22 维):
#    - 3 疾病相似度特征: q_score_Description, q_score_Pathway, q_score_Slim
#    - 7 药物相似度特征: p_score_Target, p_score_Category, p_score_Conditions,
#                         p_score_Description, p_score_Mechanism,
#                         p_score_Pharmacodynamics, p_score_Smile
#    - 2 拓扑计数特征: count_drug (=|B_s|), count_disease (=|A_d|)
#    - 3+7 交叉乘法特征: adj_q_score_* = q_score_* × count_drug
#                        adj_p_score_* = p_score_* × count_disease
#
#  [关键修复] 使用 mapping80.csv 计算邻居 (防数据泄漏)

import pandas as pd
import os
# 数据路径: 自动定位项目根 (兼容从根目录或 code/ 启动)
if os.path.exists("data"):
    DATA_ROOT = "data"
elif os.path.exists("../data"):
    DATA_ROOT = "../data"
else:
    raise FileNotFoundError("找不到 data/ 目录, 请确认工作目录为项目根")
base_path = os.path.join(DATA_ROOT, "DDCD", "SimilarityMatrices")
mapping_base = os.path.join(DATA_ROOT, "DDCD", "Mapping")


# ==========================================
# 1. 配置文件路径
# ==========================================

# --- A. 疾病相似度文件 (3个特征) ---
disease_sim_files = {
    'Description': 'diseaseDecription_bert.csv',
    'Pathway':     'diseasePathwayName_jaccard.csv',
    'Slim':        'diseaseSlimmapping_jaccard.csv'
}

# --- B. 药物相似度文件 (7个特征) ---
drug_sim_files = {
    'Target':           'drugTarget_jaccard',
    'Category':         'drugCategory_jaccard',
    'Conditions':       'drugConditions_jaccard',
    'Description':      'drugDescription_bert',
    'Mechanism':        'drugMechanism_bert',
    'Pharmacodynamics': 'drugPharmacodynamics_bert',
    'Smile':            'drugSmile_tanimoto'
}

# ==========================================
# 2. 列名定义 (恰好 22 维特征)
# ==========================================
# 疾病侧: 3 个原始 + 3 个交叉 = 6
DISEASE_FEATURE_NAMES = ['Description', 'Pathway', 'Slim']
# 药物侧: 7 个原始 + 7 个交叉 = 14
DRUG_FEATURE_NAMES = ['Target', 'Category', 'Conditions', 'Description',
                       'Mechanism', 'Pharmacodynamics', 'Smile']
# 拓扑计数: 2 个
COUNT_FEATURES = ['count_drug', 'count_disease']

# 构建最终 22 维特征列名
q_score_cols = [f'q_score_{name}' for name in DISEASE_FEATURE_NAMES]      # 3
p_score_cols = [f'p_score_{name}' for name in DRUG_FEATURE_NAMES]         # 7
adj_q_cols   = [f'adj_q_score_{name}' for name in DISEASE_FEATURE_NAMES] # 3
adj_p_cols   = [f'adj_p_score_{name}' for name in DRUG_FEATURE_NAMES]    # 7

FEATURE_22 = (COUNT_FEATURES + q_score_cols + p_score_cols +
              adj_q_cols + adj_p_cols)
assert len(FEATURE_22) == 22, f"Expected 22 features, got {len(FEATURE_22)}"

# DataFrame 列顺序: ID + 22特征
all_columns = ['drugID', 'diseaseID'] + FEATURE_22
print(f"✅ 配置完成。特征维度: {len(FEATURE_22)} (= 2计数 + 3+7原始 + 3+7交叉)")
print(f"   疾病特征 ({len(DISEASE_FEATURE_NAMES)}): {DISEASE_FEATURE_NAMES}")
print(f"   药物特征 ({len(DRUG_FEATURE_NAMES)}): {DRUG_FEATURE_NAMES}")

✅ 配置完成。特征维度: 22 (= 2计数 + 3+7原始 + 3+7交叉)
   疾病特征 (3): ['Description', 'Pathway', 'Slim']
   药物特征 (7): ['Target', 'Category', 'Conditions', 'Description', 'Mechanism', 'Pharmacodynamics', 'Smile']


In [10]:
# --- 修正后的 Cell 2: 加载 3 个疾病相似度矩阵 ---
disease_sim_dict = {}
print("正在加载 DDCD 疾病数据 (Diseases)...")

for feature_name, filename in disease_sim_files.items():
    full_path = os.path.join(base_path, filename)
    if os.path.exists(full_path):
        try:
            # 假设第一列是 ID，设为索引
            df = pd.read_csv(full_path, index_col=0)
            disease_sim_dict[feature_name] = df
            print(f"✅ 成功加载: {feature_name} (Shape: {df.shape})")
        except Exception as e:
            print(f"❌ 加载失败: {filename}, 错误: {e}")
    else:
        print(f"❌ 警告: 找不到文件 {filename}，请检查路径！")

正在加载 DDCD 疾病数据 (Diseases)...
✅ 成功加载: Description (Shape: (1573, 1573))
✅ 成功加载: Pathway (Shape: (1573, 1573))
✅ 成功加载: Slim (Shape: (1573, 1573))


In [11]:
# --- 修正后的 Cell 3: 加载药物相似度矩阵 ---
drug_sim_dict = {}
print("正在加载 DDCD 药物数据 (Drugs)...")

for feature_name, filename in drug_sim_files.items():
    full_path = os.path.join(base_path, filename)
    if os.path.exists(full_path):
        try:
            df = pd.read_csv(full_path, index_col=0)
            drug_sim_dict[feature_name] = df
            print(f"✅ 成功加载: {feature_name} (Shape: {df.shape})")
        except Exception as e:
            print(f"❌ 加载失败: {filename}, 错误: {e}")
    else:
        print(f"❌ 警告: 找不到文件 {filename}")

正在加载 DDCD 药物数据 (Drugs)...
✅ 成功加载: Target (Shape: (1410, 1410))
✅ 成功加载: Category (Shape: (1410, 1410))
✅ 成功加载: Conditions (Shape: (1410, 1410))
✅ 成功加载: Description (Shape: (1410, 1410))
✅ 成功加载: Mechanism (Shape: (1410, 1410))
✅ 成功加载: Pharmacodynamics (Shape: (1410, 1410))
✅ 成功加载: Smile (Shape: (1410, 1410))


In [12]:
# --- 加载映射文件 ---
# mapping80: 用于计算邻居 Ad, Bs (防泄漏)
# mapping_full: 用于标签分配 (ground truth)

mapping80_path = os.path.join(mapping_base, "mapping80.csv")
mapping_full_path = os.path.join(mapping_base, "mapping.csv")

# 训练用映射 (特征计算的邻居来源)
mapping_train = pd.read_csv(mapping80_path)
if 'DrugID' not in mapping_train.columns:
    mapping_train.columns = ['DrugID', 'DiseaseID'] + mapping_train.columns[2:].tolist()
mapping_train['DrugID'] = mapping_train['DrugID'].astype(str).str.strip()
mapping_train['DiseaseID'] = mapping_train['DiseaseID'].astype(str).str.strip()

# 全量映射 (仅用于标签赋值)
mapping_full = pd.read_csv(mapping_full_path)
if 'DrugID' not in mapping_full.columns:
    mapping_full.columns = ['DrugID', 'DiseaseID'] + mapping_full.columns[2:].tolist()
mapping_full['DrugID'] = mapping_full['DrugID'].astype(str).str.strip()
mapping_full['DiseaseID'] = mapping_full['DiseaseID'].astype(str).str.strip()

print(f"✅ mapping80 (特征邻居): {len(mapping_train):,} 条")
print(f"✅ mapping_full (标签真值): {len(mapping_full):,} 条")
print(f"   mapping20 (仅标签用): {len(mapping_full) - len(mapping_train):,} 条 (测试集正样本)")
mapping_train.head(3)

✅ mapping80 (特征邻居): 34,488 条
✅ mapping_full (标签真值): 42,200 条
   mapping20 (仅标签用): 7,712 条 (测试集正样本)


,DrugID,DiseaseID,numPreds
0,DB09140,MESH:D000013,40
1,DB09140,MESH:D000014,40
2,DB09140,MESH:D000799,40


In [6]:
# 从训练关联中获取所有药物和疾病 (对齐 mapping80)
all_drugs = sorted(mapping_train['DrugID'].unique().tolist())
all_diseases = sorted(mapping_train['DiseaseID'].unique().tolist())
print(f"药物数: {len(all_drugs):,} | 疾病数: {len(all_diseases):,}")
print(f"总药物-疾病对 (笛卡尔积): {len(all_drugs):,} × {len(all_diseases):,} = {len(all_drugs)*len(all_diseases):,}")

药物数: 1,410 | 疾病数: 1,573
总药物-疾病对 (笛卡尔积): 1,410 × 1,573 = 2,217,930


In [13]:
# --- Cell 6: 核心特征计算 (符合 MiRAGE 论文 Eq.3-4) ---
# 
# 论文: https://academic.oup.com/bib/article/25/4/bbae337/7717951
#
#   Eq.3: score_disease = max Sim(s', s)  for s' in Ad (drug d 的已知疾病)
#          score_drug    = max Sim(d', d)  for d' in Bs (disease s 的已知药物)
#   Eq.4: adjusted_score = score × |N|  (交叉乘法加权)
#
#   交叉乘法解释 → 22 维特征:
#     - 疾病侧: adj_q_score = q_score × count_drug (=|Bs|)
#     - 药物侧: adj_p_score = p_score × count_disease (=|Ad|)
#     含义: 疾病相似度被"该疾病有多少药物治疗"加权; 药物反之亦然
#
#   [关键修复]
#     ① 邻居 Ad, Bs 仅从 mapping80 计算 → 测试正样本不泄漏
#     ② 标签从 mapping_full 赋值 → 保留测试正样本的真实标签
#     ③ 交叉乘法内联 → 一步生成 22 维

from tqdm import tqdm
import numpy as np

# --- 预计算邻居索引 (仅用 mapping80) ---
drug_to_diseases_train = mapping_train.groupby('DrugID')['DiseaseID'].apply(set).to_dict()
disease_to_drugs_train = mapping_train.groupby('DiseaseID')['DrugID'].apply(set).to_dict()

# --- 预计算标签索引 (用全量 mapping) ---
drug_to_diseases_full = mapping_full.groupby('DrugID')['DiseaseID'].apply(set).to_dict()

rows_list = []
total_pairs = len(all_drugs) * len(all_diseases)
print(f"开始计算特征 (总对数: {len(all_drugs):,} × {len(all_diseases):,} = {total_pairs:,})...")

for drug_test in tqdm(all_drugs):
    # 邻居集合 (仅 mapping80)
    known_diseases_train = drug_to_diseases_train.get(drug_test, set())
    # 标签集合 (全量 mapping)
    known_diseases_full = drug_to_diseases_full.get(drug_test, set())

    for disease_test in all_diseases:
        # 邻居 Bs (仅 mapping80)
        known_drugs_train = disease_to_drugs_train.get(disease_test, set())

        # 排除自身
        Ad = known_diseases_train - {disease_test}
        Bs = known_drugs_train - {drug_test}

        count_disease = len(Ad)  # |Ad|
        count_drug = len(Bs)     # |Bs|

        # --- 疾病侧特征 q_score & adj_q_score (3+3=6 维) ---
        q_feats = {}
        adj_q_feats = {}
        for fname, mat in disease_sim_dict.items():
            val = 0.0
            if Ad and disease_test in mat.index:
                valid = [d for d in Ad if d in mat.index]
                if valid:
                    v = mat.loc[disease_test, valid].max()
                    if not pd.isna(v):
                        val = float(v)
            q_feats[f'q_score_{fname}'] = val
            adj_q_feats[f'adj_q_score_{fname}'] = val * count_drug

        # --- 药物侧特征 p_score & adj_p_score (7+7=14 维) ---
        p_feats = {}
        adj_p_feats = {}
        for fname, mat in drug_sim_dict.items():
            val = 0.0
            if Bs and drug_test in mat.index:
                valid = [d for d in Bs if d in mat.index]
                if valid:
                    v = mat.loc[drug_test, valid].max()
                    if not pd.isna(v):
                        val = float(v)
            p_feats[f'p_score_{fname}'] = val
            adj_p_feats[f'adj_p_score_{fname}'] = val * count_disease

        # --- 组装 (ID + 22特征 + label) ---
        row = {
            'drugID': drug_test,
            'diseaseID': disease_test,
            'count_drug': count_drug,
            'count_disease': count_disease,
        }
        row.update(q_feats)       #  3 cols
        row.update(p_feats)       #  7 cols
        row.update(adj_q_feats)   #  3 cols
        row.update(adj_p_feats)   #  7 cols
        # 标签: 来自全量 mapping (含 mapping20 测试正样本)
        row['label'] = 1 if disease_test in known_diseases_full else 0

        rows_list.append(row)

# 构建 DataFrame
df_scores = pd.DataFrame(rows_list)
df_scores.fillna(0.0, inplace=True)

# 验证
feature_cols_only = [c for c in df_scores.columns if c not in ['drugID', 'diseaseID', 'label']]
n_pos = (df_scores['label'] == 1).sum()
n_neg = (df_scores['label'] == 0).sum()
print(f"\n✅ 计算完成!")
print(f"   总行数: {len(df_scores):,}")
print(f"   特征维度: {len(feature_cols_only)} (预期 22)")
assert len(feature_cols_only) == 22, f"特征维度错误! 预期 22, 实际 {len(feature_cols_only)}"
print(f"   正样本: {n_pos:,} ({n_pos/len(df_scores)*100:.2f}%)")
print(f"   负样本: {n_neg:,} ({n_neg/len(df_scores)*100:.2f}%)")
print(f"   特征列: {feature_cols_only[:5]} ... {feature_cols_only[-3:]}")
df_scores.head(3)

开始计算特征 (总对数: 1,410 × 1,573 = 2,217,930)...


100%|██████████| 1410/1410 [1:04:34<00:00,  2.75s/it]



✅ 计算完成!
   总行数: 2,217,930
   特征维度: 22 (预期 22)
   正样本: 42,200 (1.90%)
   负样本: 2,175,730 (98.10%)
   特征列: ['count_drug', 'count_disease', 'q_score_Description', 'q_score_Pathway', 'q_score_Slim'] ... ['adj_p_score_Mechanism', 'adj_p_score_Pharmacodynamics', 'adj_p_score_Smile']


,drugID,diseaseID,count_drug,count_disease,q_score_Description,q_score_Pathway,q_score_Slim,p_score_Target,p_score_Category,p_score_Conditions,...,adj_q_score_Pathway,adj_q_score_Slim,adj_p_score_Target,adj_p_score_Category,adj_p_score_Conditions,adj_p_score_Description,adj_p_score_Mechanism,adj_p_score_Pharmacodynamics,adj_p_score_Smile,label
0,DB00006,MESH:D000013,21,10,0.647925,0.136555,0.0,0.006803,0.040000,0.000000,...,2.867647,0.0,0.068027,0.400000,0.000000,8.967041,8.416816,7.697048,0.720721,0
1,DB00006,MESH:D000014,68,10,0.561902,0.152381,0.0,0.000000,0.133333,0.017241,...,10.361905,0.0,0.000000,1.333333,0.172414,8.667165,8.420105,7.958895,1.238739,0
2,DB00006,MESH:D000015,22,10,0.646850,0.334454,0.0,0.000000,0.038462,0.000000,...,7.357983,0.0,0.000000,0.384615,0.000000,8.278577,9.100370,7.777154,0.956341,0


In [14]:
# --- Cell 7: 保存结果 ---
# 交叉乘法已在 Cell 6 中内联计算 (符合论文 Eq.4)
# 此处直接保存，不再做后处理

output_file = "results/MiRAGE_score_DDCD.csv"
os.makedirs("results", exist_ok=True)
df_scores.to_csv(output_file, index=False)

print(f"✅ 结果已保存至 {output_file}")
print(f"   总行数: {len(df_scores):,}")
print(f"   总列数: {len(df_scores.columns)} (= drugID + diseaseID + label + 22特征)")
print(f"   正样本: {(df_scores['label']==1).sum():,}")
print(f"   负样本: {(df_scores['label']==0).sum():,}")
print(f"\n前 5 行预览:")
df_scores.head()

✅ 结果已保存至 results/MiRAGE_score_DDCD.csv
   总行数: 2,217,930
   总列数: 25 (= drugID + diseaseID + label + 22特征)
   正样本: 42,200
   负样本: 2,175,730

前 5 行预览:


,drugID,diseaseID,count_drug,count_disease,q_score_Description,q_score_Pathway,q_score_Slim,p_score_Target,p_score_Category,p_score_Conditions,...,adj_q_score_Pathway,adj_q_score_Slim,adj_p_score_Target,adj_p_score_Category,adj_p_score_Conditions,adj_p_score_Description,adj_p_score_Mechanism,adj_p_score_Pharmacodynamics,adj_p_score_Smile,label
0,DB00006,MESH:D000013,21,10,0.647925,0.136555,0.0,0.006803,0.040000,0.000000,...,2.867647,0.0,0.068027,0.400000,0.000000,8.967041,8.416816,7.697048,0.720721,0
1,DB00006,MESH:D000014,68,10,0.561902,0.152381,0.0,0.000000,0.133333,0.017241,...,10.361905,0.0,0.000000,1.333333,0.172414,8.667165,8.420105,7.958895,1.238739,0
2,DB00006,MESH:D000015,22,10,0.646850,0.334454,0.0,0.000000,0.038462,0.000000,...,7.357983,0.0,0.000000,0.384615,0.000000,8.278577,9.100370,7.777154,0.956341,0
3,DB00006,MESH:D000022,8,10,0.405862,0.472581,0.0,0.000000,0.031250,0.000000,...,3.780645,0.0,0.000000,0.312500,0.000000,8.278577,8.336291,7.691096,0.946502,0
4,DB00006,MESH:D000026,1,10,0.534620,0.239130,0.0,0.000000,0.022222,0.000000,...,0.239130,0.0,0.000000,0.222222,0.000000,7.742505,6.710027,5.657667,0.617284,0


In [15]:
# 依赖检查
import pandas as pd
import numpy as np
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print("✅ 依赖正常")

pandas : 2.2.2
numpy  : 1.26.4
✅ 依赖正常


# 